In [3]:
pip install PyPortfolioOpt

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 39.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 43.5 MB/s  0:00:00 eta 0:00:01
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [PyPortfolioOpt]m [cvxpy]]base]
Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install quantstats

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [quantstats]
Note: you may need to restart the kernel to use updated packages.


In [19]:
# %matplotlib inline
%config InlineBackend.figure_format = "retina"
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
})

import seaborn as sns
sns.set_theme(context="talk", style="whitegrid", 
              palette="colorblind", color_codes=True, 
              rc={"figure.figsize": [12, 8]})

import yfinance as yf
import numpy as np
import pandas as pd
import quantstats as qs

import requests
import csv

from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

In [ ]:
# Carteira teorica do Idiv

url = "https://raw.githubusercontent.com/BDonadelli/Finance-playground/main/data/Cart_Idiv.csv"

response = requests.get(url)
response.encoding = 'utf-8'  # ou 'latin-1' se necessário

# Divide o conteúdo em linhas
linhas = response.text.splitlines()

# Ignora as duas primeiras e duas últimas linhas
linhas_filtradas = linhas[2:-2]

# Extrai a primeira coluna de cada linha restante
ASSETS = []
for linha in linhas_filtradas:
    # O separador é ";", pega o primeiro campo
    campos = linha.split(';')
    if campos:  # garante que a linha não está vazia
        ASSETS.append((campos[0],campos[4]))

ASSETS = [(ticker, float(str(value).replace(',', '.'))) for ticker, value in ASSETS]

try:
    pesos_indice = [value for ticker, value in ASSETS]
    tickers = [ticker+'.SA' for ticker, value in ASSETS]
    tickers.sort()
except :    
    tickers = [ticker+'.SA' for ticker in ASSETS]

tickers.sort()

#### Parâmetros

In [21]:
rf = 0.14               # taxa livre de risco
n_days=252              # dias no ano do calendario financeiro, assumindo dados diários pegos no Yahoo Finance
#n_monte_carlo = 10**6 # quantidade de carteiras na simulação

# Definição do período e download dos dados
data_inicio = '2018-01-01'
data_fim = '2026-04-30'

#### preços de fechamento

Baixa dados e limpa a base

In [24]:
prices = yf.download(tickers, start=data_inicio, end=data_fim , auto_adjust=True)['Close']
prices.columns = [col.replace('.SA', '') for col in prices.columns]

benchm = yf.download('^BVSP', start=data_inicio, end=data_fim , auto_adjust=True)['Close']

[*********************100%***********************]  52 of 52 completed
[*********************100%***********************]  1 of 1 completed


In [25]:
# Empresas com mais de 'limiar' dados faltantes
limiar = 10
missing = prices.isna().sum()
empresas_missing = missing[missing > 10].index.tolist()

# print(empresas_missing)
print(missing[missing > 10].sort_values(ascending=False))

BRBI11    872
RECV3     824
CXSE3     821
CMIN3     774
CURY3     674
LAVV3     663
PGMN3     663
ALOS3     394
LOGG3     242
VBBR3      49
dtype: int64


In [26]:
import plotly.express as px

if empresas_missing:
    fig = px.line(
        prices[empresas_missing].reset_index(),
        x='Date',
        y=empresas_missing,
        title='Empresas com mais de 10 dados faltantes',
        labels={'value': 'Preço', 'Date': 'Data', 'variable': 'Empresa'}
    )

    fig.show()

In [32]:
# mantem colunas (axis=1) ue possuem no mínimo len(prices) - 20 valores não-nulos.
prices = prices.dropna(axis=1, thresh=len(prices) - 10)
# preenche dados faltantes repetindo ultimo valor
prices = prices.ffill()

prices

,ABCB4,AGRO3,BBAS3,BBDC3,BBDC4,BBSE3,BRAP4,BRSR6,CMIG4,CPFE3,...,SAPR11,SLCE3,SYNE3,TAEE11,TGMA3,TIMS3,UNIP6,VALE3,VLID3,VULC3
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,9.208961,6.842369,9.178532,11.138394,12.179829,13.367786,4.202160,7.811340,1.498885,10.347329,...,12.398588,3.451246,0.866539,9.415704,11.868073,7.978252,5.058344,21.669315,12.036351,5.375475
2018-01-03,9.247841,6.836968,9.295600,11.185779,12.235813,13.377101,4.241883,7.848087,1.485962,10.219522,...,12.291394,3.481202,0.913177,9.428870,11.868073,7.984300,5.428318,21.539465,12.218149,5.641646
2018-01-04,9.220068,7.058385,9.384793,11.389482,12.436572,13.405049,4.362473,7.895332,1.468734,10.144964,...,12.192606,3.467472,0.951421,9.279690,12.324996,7.948033,5.370699,21.627764,12.218149,5.786304
2018-01-05,9.353371,7.204199,9.384793,11.392924,12.507016,13.493545,4.467455,7.998999,1.470888,9.979875,...,12.285086,3.532377,0.951421,9.323565,12.461476,8.014520,5.455612,21.965376,12.193076,5.786304
2018-01-08,9.442240,7.236602,9.407092,11.392924,12.503494,13.572727,4.545482,8.132938,1.475195,10.139639,...,12.211523,3.513655,0.960748,9.279690,12.461476,7.905723,5.701250,22.453606,12.161732,5.815234
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-23,25.160000,19.680000,23.000000,17.191864,19.949961,34.270000,24.059999,16.155130,13.088566,50.148193,...,42.849998,17.530001,4.060000,43.706520,32.939999,26.049999,60.869999,85.970001,19.760000,16.120001
2026-04-24,25.280001,19.850000,22.700001,17.091970,19.900011,34.290001,23.950001,15.876423,12.804031,49.388653,...,43.610001,17.370001,4.000000,43.168262,32.230000,26.100000,60.540001,85.870003,19.850000,15.940000
2026-04-27,24.930000,19.139999,22.510000,16.952118,19.710201,33.950001,23.850000,15.687300,12.558743,48.906994,...,41.549999,17.240000,3.940000,42.678936,32.250000,25.770000,59.880001,85.500000,19.440001,15.780000


In [28]:
returns = prices.pct_change().dropna()

print(len(prices),len(returns),len(tickers),len(ASSETS),len(prices.columns))

2068 2067 52 52 42


Matriz covariância por Ledoit Wolf

In [ ]:
from pypfopt import black_litterman, risk_models
from sklearn.covariance import LedoitWolf

"""
cov_matrix é uma NxN matriz de covariância
tickers é um dicionário com os ativos
market_prices é uma série de preços de ativos
"""

tickers = [item for item in ASSETS]
market_prices = [item[1] for item in ASSETS]

#cov = np.cov(market_prices)

model = LedoitWolf()
cov_matrix = model.fit(returns).covariance_
df_cov_matrix = pd.DataFrame(cov_matrix, index=returns.columns, columns=returns.columns)

print(df_cov_matrix)

           ABCB4     AGRO3     BBAS3     BBDC3     BBDC4     BBSE3     BRAP4  \
ABCB4   0.000378  0.000102  0.000247  0.000227  0.000231  0.000115  0.000121   
AGRO3   0.000102  0.000324  0.000104  0.000095  0.000092  0.000059  0.000084   
BBAS3   0.000247  0.000104  0.000503  0.000338  0.000340  0.000175  0.000159   
BBDC3   0.000227  0.000095  0.000338  0.000442  0.000428  0.000157  0.000166   
BBDC4   0.000231  0.000092  0.000340  0.000428  0.000469  0.000149  0.000162   
BBSE3   0.000115  0.000059  0.000175  0.000157  0.000149  0.000251  0.000084   
BRAP4   0.000121  0.000084  0.000159  0.000166  0.000162  0.000084  0.000479   
BRSR6   0.000218  0.000087  0.000298  0.000262  0.000266  0.000133  0.000135   
CMIG4   0.000198  0.000091  0.000285  0.000252  0.000248  0.000148  0.000132   
CPFE3   0.000128  0.000068  0.000167  0.000154  0.000152  0.000098  0.000076   
CSMG3   0.000162  0.000078  0.000211  0.000183  0.000181  0.000109  0.000096   
DIRR3   0.000232  0.000125  0.000262  0.

In [33]:
delta = black_litterman.market_implied_risk_aversion(prices)
prior = black_litterman.market_implied_prior_returns(tickers, delta, cov_matrix)

/home/caio/Documentos/ic_26/.venv/lib/python3.12/site-packages/pypfopt/black_litterman.py:45: RuntimeWarning: If cov_matrix is not a dataframe, market cap index must be aligned to cov_matrix
  warnings.warn(


ValueError: operands could not be broadcast together with shapes (52,) (104,) 